# Stellar mass function: how $\tau_0$ moves mass between centrals and satellites

The dynamical-friction timescale $\tau_0$ is a *book-keeping* knob: it does not change the total stellar mass produced (set by gas physics, feedback and cooling) but only **where that mass ends up** — in centrals (BCGs) or in satellites. This makes the SMF a uniquely diagnostic test:

* The **total SMF** should be approximately invariant under changes of $\tau_0$, modulo small second-order effects (e.g. gas brought in by the merging satellite enabling a bit more star formation in the central).
* The **central SMF** should bracket between two extremes:
  - $\tau_0 = 0$: every satellite immediately joins the central, so the central SMF gains mass at the high-$M_\star$ end.
  - $\tau_0 \to \infty$: no satellites merge, so the central SMF is suppressed at the high-$M_\star$ end.
* The **satellite SMF** should move oppositely: enhanced at $\tau_0 \to \infty$, suppressed at $\tau_0 = 0$.

If the dynamical-friction timescale matters, the central and satellite SMFs should split visibly at $\log_{10}(M_\star/[M_\odot/h]) \gtrsim 11$ — exactly the regime of BCGs and most-massive satellites.

All masses are in $M_\odot/h$.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from utils.matplotlib_config import setconfig

sys.path.insert(0, str(Path.cwd()))
from tau0_helpers import (
    DEFAULT_IVOLS,
    RUNS,
    RUN_LABELS,
    SNAPSHOTS,
    collect_per_ivol,
    safe_ratio,
    smf_split_per_ivol,
    stack_per_ivol,
    style_for,
)

setconfig({
    'figure': {'figsize': (8, 6)},
    'font': {'size': 14},
    'legend': {'fontsize': 12, 'title_fontsize': 13},
    'axes': {'labelsize': 14},
})

# Slightly extended bin grid into the BCG-mass tail (where the τ₀ split should be largest).
mstar_bins = np.arange(8.0, 12.61, 0.15)
mstar_centers = 0.5 * (mstar_bins[1:] + mstar_bins[:-1])
ivols = DEFAULT_IVOLS

summarise_smf = lambda d: smf_split_per_ivol(d, mstar_bins)

## Compute per-ivol SMF (total / cen / sat) for every (run, snapshot)

In [ ]:
stacked = {}
for snapshot in SNAPSHOTS:
    stacked[snapshot] = {}
    for run_label, run_path in RUNS.items():
        summaries = collect_per_ivol(run_path, snapshot, ivols, summarise_smf)
        stacked[snapshot][run_label] = stack_per_ivol(
            summaries,
            keys=('phi_total', 'phi_cen', 'phi_sat'),
            nboot=500,
            seed=29,
        )
        n = stacked[snapshot][run_label]['n_used']
        print(f"{snapshot} {run_label:>10s}: {n:2d} ivols used")

## Figure 1 — Total / central / satellite SMF, $z = 0$ and $z = 0.5$

Each row is a snapshot; the three columns are the three SMF components. Bands show bootstrap 16-84 ranges over ivols. The y-axis is $\log_{10}\Phi$ in $\mathrm{Mpc}^{-3}\,h^3\,\mathrm{dex}^{-1}$, so all three runs being visually overlaid in the leftmost (total) panel is the *expected* signature of $\tau_0$ being a redistribution knob.

The headline science is the **central** column at high $M_\star$: the gap between $\tau_0=0$ and $\tau_0=\infty$ at $\log_{10}M_\star \gtrsim 11.0$ is the size of the BCG-mass effect.

In [ ]:
components = [
    ('phi_total', 'Total'),
    ('phi_cen', 'Centrals'),
    ('phi_sat', 'Satellites'),
]

fig, axes = plt.subplots(
    len(SNAPSHOTS), len(components),
    figsize=(13, 4.6 * len(SNAPSHOTS)),
    sharex=True, sharey=True,
)
if len(SNAPSHOTS) == 1:
    axes = axes[None, :]

for i, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    for j, (key, comp_label) in enumerate(components):
        ax = axes[i, j]
        for run_label in RUNS:
            s = stacked[snapshot][run_label]
            if s['n_used'] == 0 or key not in s:
                continue
            phi = s[key]['mean']
            lo = s[key]['boot_lo']
            hi = s[key]['boot_hi']
            ok = np.isfinite(phi) & (phi > 0)
            st = style_for(run_label)
            ax.plot(mstar_centers[ok], phi[ok], '-', lw=2.2, **st)
            band_ok = ok & (lo > 0) & (hi > 0)
            # ax.fill_between(
            #     mstar_centers[band_ok], lo[band_ok], hi[band_ok],
            #     color=st['color'], alpha=0.20, linewidth=0,
            # )
        ax.set_yscale('log')
        ax.set_ylim(1e-6, 0.2)
        ax.set_xlim(8.5, 12.4)
        #ax.grid(True, alpha=0.3, which='both')
        if i == 0:
            ax.set_title(comp_label)
        if i == len(SNAPSHOTS) - 1:
            ax.set_xlabel(r'$\log_{10}\,M_\star\ [M_\odot/h]$')
        if j == 0:
            ax.set_ylabel(r'$\Phi\ [(\mathrm{Mpc}/h)^{-3}\,\mathrm{dex}^{-1}]$' + f'\nz = {z_val:.1f}')
        if i == 0 and j == 0:
            ax.legend(loc='lower left', fontsize=10)
fig.suptitle(r'Stellar mass function across $\tau_0$ variants', y=1.0)
plt.tight_layout()
plt.show()

## Figure 2 — Ratio panels relative to Default

Linear-axis ratios make the high-mass behaviour easier to read. We focus on $\log_{10}M_\star \geq 9.5$ (above which we trust the Default SMF, given the box-resolution floor).

In [ ]:
fig, axes = plt.subplots(
    len(SNAPSHOTS), len(components),
    figsize=(13, 4.0 * len(SNAPSHOTS)),
    sharex=True, sharey=True,
)
if len(SNAPSHOTS) == 1:
    axes = axes[None, :]

for i, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    for j, (key, comp_label) in enumerate(components):
        ax = axes[i, j]
        s_def = stacked[snapshot]['Default']
        if s_def['n_used'] == 0 or key not in s_def:
            continue
        phi_def = s_def[key]['mean']
        for run_label in RUNS:
            if run_label == 'Default':
                continue
            s = stacked[snapshot][run_label]
            if s['n_used'] == 0 or key not in s:
                continue
            ratio = safe_ratio(s[key]['mean'], phi_def)
            ratio_lo = safe_ratio(s[key]['boot_lo'], phi_def)
            ratio_hi = safe_ratio(s[key]['boot_hi'], phi_def)
            ok = np.isfinite(ratio) & (mstar_centers >= 9.5) & (phi_def > 1e-7)
            st = style_for(run_label)
            ax.plot(mstar_centers[ok], ratio[ok], '-', lw=2.2, **st)
            ax.fill_between(
                mstar_centers[ok], ratio_lo[ok], ratio_hi[ok],
                color=st['color'], alpha=0.20, linewidth=0,
            )
        ax.axhline(1.0, color='0.4', ls=':', lw=1)
        ax.set_yscale('log')
        ax.set_ylim(0.05, 20.0)
        ax.set_yticks([0.1, 0.3, 1.0, 3.0, 10.0])
        ax.set_yticklabels(['0.1', '0.3', '1', '3', '10'])
        ax.set_xlim(9.5, 12.3)
        ax.grid(True, alpha=0.3, which='both')
        if i == 0:
            ax.set_title(comp_label)
        if i == len(SNAPSHOTS) - 1:
            ax.set_xlabel(r'$\log_{10}\,M_\star\ [M_\odot/h]$')
        if j == 0:
            ax.set_ylabel(r'$\Phi/\Phi^\mathrm{default}$' + f'\nz = {z_val:.1f}')
        if i == 0 and j == 0:
            ax.legend(loc='upper left', fontsize=10)
fig.suptitle(r'SMF ratios relative to default $\tau_0$', y=1.0)
plt.tight_layout()
plt.show()

## Figure 3 — Satellite fraction $\Phi_\mathrm{sat}\,/\,\Phi_\mathrm{total}$

Independent of normalisation, the satellite fraction at fixed $M_\star$ is the cleanest direct readout of how $\tau_0$ partitions the population.

In [ ]:
fig, axes = plt.subplots(1, len(SNAPSHOTS), figsize=(11, 4.5), sharex=True, sharey=True)
if len(SNAPSHOTS) == 1:
    axes = [axes]

for j, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    ax = axes[j]
    for run_label in RUNS:
        s = stacked[snapshot][run_label]
        if s['n_used'] == 0:
            continue
        f_sat = safe_ratio(s['phi_sat']['mean'], s['phi_total']['mean'])
        f_sat_lo = safe_ratio(s['phi_sat']['boot_lo'], s['phi_total']['mean'])
        f_sat_hi = safe_ratio(s['phi_sat']['boot_hi'], s['phi_total']['mean'])
        ok = np.isfinite(f_sat) & (s['phi_total']['mean'] > 1e-7) & (mstar_centers >= 9.0)
        st = style_for(run_label)
        ax.plot(mstar_centers[ok], f_sat[ok], '-', lw=2.2, **st)
        ax.fill_between(
            mstar_centers[ok], f_sat_lo[ok], f_sat_hi[ok],
            color=st['color'], alpha=0.20, linewidth=0,
        )
    ax.set_xlabel(r'$\log_{10}\,M_\star\ [M_\odot/h]$')
    ax.set_xlim(9.0, 12.3)
    ax.set_ylim(0.0, 1.0)
    ax.set_title(f'{snapshot}  (z = {z_val:.1f})')
    ax.grid(True, alpha=0.3)
    if j == 0:
        ax.set_ylabel(r'$f_\mathrm{sat}\ \equiv\ \Phi_\mathrm{sat}/\Phi_\mathrm{total}$')
        ax.legend(loc='upper right', fontsize=10)
fig.suptitle(r'Satellite fraction at fixed stellar mass', y=1.02)
plt.tight_layout()
plt.show()

## Figure 4 — Cumulative number density above $M_\star$

$N(>M_\star) = \int_{M_\star}^\infty \Phi(M)\,d\log M$. At the high-$M_\star$ end this isolates the "abundance of BCGs above a given mass" — the most observationally-relevant statistic for cluster surveys.

In [ ]:
dlog = np.diff(mstar_bins)

def cumulative_above(phi):
    return np.cumsum((phi * dlog)[::-1])[::-1]

fig, axes = plt.subplots(1, len(SNAPSHOTS), figsize=(11, 4.8), sharex=True, sharey=True)
if len(SNAPSHOTS) == 1:
    axes = [axes]

for j, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    ax = axes[j]
    for run_label in RUNS:
        s = stacked[snapshot][run_label]
        if s['n_used'] == 0:
            continue
        n_cum_cen = cumulative_above(np.where(np.isfinite(s['phi_cen']['mean']), s['phi_cen']['mean'], 0))
        st = style_for(run_label)
        ax.plot(mstar_centers, n_cum_cen, '-', lw=2.2, **st)
    ax.set_yscale('log')
    ax.set_xlim(10.5, 12.3)
    ax.set_ylim(1e-6, 1e-2)
    ax.set_xlabel(r'$\log_{10}\,M_\star\ [M_\odot/h]$')
    ax.set_title(f'{snapshot}  (z = {z_val:.1f})')
    ax.grid(True, alpha=0.3, which='both')
    if j == 0:
        ax.set_ylabel(r'$N(>M_\star)\ [(\mathrm{Mpc}/h)^{-3}]$ \textit{(centrals)}')
        ax.legend(loc='upper right', fontsize=10)
fig.suptitle(r'Cumulative central abundance — sensitive to BCG accretion', y=1.02)
plt.tight_layout()
plt.show()

## Quantitative summary

Numerical bracket on the $\tau_0$ effect at three diagnostic stellar masses.
Use $\Phi^\mathrm{model}/\Phi^\mathrm{default}$ (a redistribution-only effect should leave the total ratio close to 1 at every mass).

In [ ]:
import pandas as pd
rows = []
for snapshot in SNAPSHOTS:
    z_val = SNAPSHOTS[snapshot][1]
    for log_ms_target in (10.0, 11.0, 11.5):
        i = int(np.argmin(np.abs(mstar_centers - log_ms_target)))
        s_def = stacked[snapshot]['Default']
        for run_label in RUNS:
            s = stacked[snapshot][run_label]
            if s['n_used'] == 0:
                continue
            row = {'snapshot': snapshot, 'z': z_val, 'log10_Mstar': log_ms_target, 'run': run_label}
            for key in ('phi_total', 'phi_cen', 'phi_sat'):
                if key not in s or key not in s_def:
                    continue
                phi = s[key]['mean'][i]
                phi_def = s_def[key]['mean'][i]
                row[key] = f'{phi:.3e}'
                row[f'{key}/default'] = round(phi / phi_def, 3) if phi_def and np.isfinite(phi_def) and phi_def > 0 else np.nan
            rows.append(row)
df = pd.DataFrame(rows)
df

## Interpretation

What to look for in each figure:

* **Total SMF (Fig 1, leftmost column)** should be nearly invariant across $\tau_0$ runs — confirming that $\tau_0$ is a redistribution knob and that the *total* stellar-mass production isn't significantly affected by where the satellite ends up.
* **Central SMF (middle column)** is where the action is. The $\tau_0=\infty$ run should drop sharply at $\log_{10}M_\star \gtrsim 11$ because BCGs never accrete satellites. The $\tau_0=0$ run should match or slightly exceed the Default at high $M_\star$.
* **Satellite SMF (right column)** should mirror the central panel: enhanced at high $M_\star$ for $\tau_0=\infty$ (massive satellites that should have merged but didn't), suppressed for $\tau_0=0$.
* **$f_\mathrm{sat}$ (Fig 3)** rising sharply with $M_\star$ for $\tau_0=\infty$, falling to zero for $\tau_0=0$. The crossover scale is the regime where $\tau_0$ matters most.
* **Cumulative central abundance (Fig 4)** is the BCG-counts statistic — directly comparable to cluster-survey counts above a given BCG mass.